# CMPT 318 — Cybersecurity
## Assignment 1: From risk estimates to a defensible decision
**Fall 2026 · 100 marks · Group assignment · Revised version 3 - plots and short definitions**

**Group number:** …  
**Names and student numbers:** …

Advise a fictional nonprofit on prevention versus recovery. Your code implements the risk model; your written analysis must explain what its results support, what they depend on, and what remains uncertain.

**Submit one file only:** `CMPT318_A1_GroupXX.ipynb`, executed and saved with outputs. Write **all explanations, calculations, interpretations, and the final 200–250-word management memo in this notebook's Markdown cells**. Do not submit a separate PDF, Pages document, report, CSV, or ZIP. The grader will use the original supplied CSV files.

**Your coding:** complete the substantive risk-analysis functions and the control-selection logic. Imports, CSV loading, repetitive experiment loops, table presentation, and plotting scaffolds are supplied. Cells tagged `student-code` and `student-analysis` identify your work. Keep the given inputs and checks.

**Working folder:** extract the supplied package and keep this notebook beside the `data` folder. Do not recreate the CSV records inside the notebook. The supplied loader reads them for you.

Due date, course weight, group arrangements, and collaboration/AI-use rules are as announced on the course page. Estimated effort: **6–8 hours per student**. Aim for no more than **1,100 words** of written analysis, including the final memo; tables, equations, and references are excluded.

The accompanying handout is a reading/editing copy of the instructions, not a student deliverable. Every assessed task is stated here. Use NumPy, pandas, Matplotlib, and the standard library. No SciPy, distribution fitting, Bayesian inference, or external data collection is required.

**Visual support:** input distributions, conditional-severity histogram, Monte Carlo stabilization, and baseline-versus-stress exceedance curves are provided. These support the existing tasks; there are no additional marks or written submissions.


### 0. Setup and file loading — provided, not assessed
Use the course Python environment. This notebook does not install or upgrade packages. Set `GROUP_NUMBER`; no other setup code needs to be written. Incomplete analytical functions raise `NotImplementedError`; supplied runners report **NOT COMPLETED** rather than supplying invented results.

In [ ]:
from pathlib import Path
from statistics import NormalDist
import sys
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, StrMethodFormatter
from IPython.display import display

GROUP_NUMBER = 0  # Replace with your assigned group number.
SEED = 3_182_026 + GROUP_NUMBER
N_SIM = 100_000
STRATEGIES = ('current', 'prevention', 'recovery')
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.3f}'.format

# Supplied runner: only intentional unfinished-work errors are caught.
def run_task(label, function):
    try:
        return function()
    except NotImplementedError as exc:
        print(f'NOT COMPLETED — {label}: {exc}')
        return None

print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | '
      f'pandas {pd.__version__} | Matplotlib {matplotlib.__version__}')
print(f'Group {GROUP_NUMBER}; seed {SEED}; main simulation size {N_SIM:,}')
NOTEBOOK_CSS = '<style>\n.jp-RenderedHTMLCommon, .jp-RenderedMarkdown, .text_cell_render, .rendered_html,\n.jp-OutputArea-output table, .dataframe {\n  font-family: Helvetica, "Nimbus Sans", Arial, sans-serif;\n  font-weight: 400; line-height: 1.35;\n}\n.jp-RenderedHTMLCommon h1, .jp-RenderedHTMLCommon h2, .jp-RenderedHTMLCommon h3,\n.text_cell_render h1, .text_cell_render h2, .text_cell_render h3,\n.rendered_html h1, .rendered_html h2, .rendered_html h3 {\n  font-family: Helvetica, "Nimbus Sans", Arial, sans-serif;\n  font-weight: 400; color: #3c8a9e;\n}\n.dataframe { border-collapse: collapse; }\n.dataframe th, .dataframe td { padding: 6px 9px; }\n</style>'

# Typography: Helvetica Regular, matching the original handout.
from IPython.display import HTML
display(HTML(NOTEBOOK_CSS))
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Helvetica', 'Nimbus Sans', 'Arial', 'DejaVu Sans'],
    'font.weight': 'normal', 'axes.labelweight': 'normal',
    'axes.titleweight': 'normal', 'figure.titleweight': 'normal',
    'font.size': 10.5, 'axes.spines.top': False, 'axes.spines.right': False,
})


### Case and decision rule
**Cedar Grove Community Services** is a fictional nonprofit with an online client portal. A material ransomware/data-extortion incident can cause an outage and response costs. Management can retain current controls, add a **prevention** package, or add a **recovery** package. The prevention package strengthens access controls; the recovery package improves restoration and incident response. The two new packages cannot be combined within the CAD 20,000 annual budget.

This is a **new, simplified teaching case**, not a reuse of the numerical assumptions in either tutorial. All amounts are CAD. The horizon is one year. Control effects are assumptions, not measured product performance.

For the case, management's rule is: choose the affordable strategy with the **lowest expected annual total cost**, subject to **P(annual incident loss > CAD 400,000) ≤ 1%**. If no strategy satisfies the rule, say so; do not silently relax it.

The annual control cost is paid even in years without an incident. It belongs in total cost, not in the incident-loss exceedance test.

The figures are supplied assumptions, not observations of a real nonprofit or vendor. The calibration CSV contains separate synthetic practice exercises; it must not be used to fit the incident-duration model.

In [ ]:
# PROVIDED: input files are distributed separately, not generated in this notebook.
DATA_DIR = Path('data')
required_files = ['strategy_parameters.csv', 'calibration_records.csv', 'case_settings.csv']
missing = [str(DATA_DIR / name) for name in required_files if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        'Extract the complete assignment ZIP and open the notebook next to its data folder. '
        'Missing: ' + ', '.join(missing)
    )
parameters = pd.read_csv(DATA_DIR / 'strategy_parameters.csv').set_index('strategy')
calibration = pd.read_csv(DATA_DIR / 'calibration_records.csv')
case_settings = pd.read_csv(DATA_DIR / 'case_settings.csv')
if len(case_settings) != 1:
    raise ValueError('case_settings.csv must contain exactly one data row.')
settings = case_settings.iloc[0]
BUDGET = float(settings['annual_budget_cad'])
LOSS_THRESHOLD = float(settings['incident_loss_threshold_cad'])
RISK_LIMIT = float(settings['annual_exceedance_limit'])
DAY_COST = float(settings['downtime_cost_cad_per_day'])
SECONDARY_COST = float(settings['secondary_cost_cad'])
SECONDARY_PROB = float(settings['secondary_probability'])
LONG_OUTAGE_PROB = float(settings['stress_long_outage_probability'])
Q_HIGH = float(settings['stress_secondary_probability_high'])

assert set(parameters.index) == set(STRATEGIES)
assert np.isfinite(parameters.to_numpy(dtype=float)).all()
assert parameters['p_incident'].between(0, 1).all()
assert (parameters['response_min'] <= parameters['response_mode']).all()
assert (parameters['response_mode'] <= parameters['response_max']).all()
assert ((parameters['outage_p50'] > 0) &
        (parameters['outage_p90'] > parameters['outage_p50'])).all()
assert len(calibration) == 10 and not calibration.isna().any().any()
assert 0 <= RISK_LIMIT <= 1 and 0 <= SECONDARY_PROB <= 1

print('Synthetic scenario parameters (costs in CAD; outage duration in days):')
display(parameters)
print('Calibration exercise records (hours, not days):')
display(calibration)
print('Case settings:')
display(case_settings.T.rename(columns={0: 'value'}))

### Short definitions for Part 1
- **Calibration:** over many comparable estimates, stated 90% intervals contain the answer about 90% of the time.
- **Coverage:** the fraction of answers inside their stated intervals; a **miss** is an answer outside. Here, endpoints count as inside.
- **Interval width:** upper endpoint minus lower endpoint. **Mean width** averages those widths over the exercises.
- **Estimate / assumption / policy:** an estimate describes an unknown quantity; a modelling assumption specifies how the model works; a policy states a management requirement. A policy is not an empirical finding.

## 1. Audit the estimates [15 marks]
**1a [4].** Classify the 18% annual incident probability, independence of outage duration and secondary response, and the board's 1% risk limit as an **estimate**, a **modelling assumption**, or a **policy choice**. For the first two, identify specific evidence that could support or challenge the input.

**1b [5].** Complete `calibration_metrics`. For each estimator, calculate inclusive interval coverage, misses, and mean width in hours. The provided runner constructs and displays the two-row table. Both estimators claimed 90% intervals before the outcomes were known.

**1c [6].** Which results suggest overconfidence? Why does agreement with the 90% target on ten exercises not establish calibration? Why are intervals from 0 to 100 hours not automatically better? Explain why an outage 90th percentile is not a maximum and why more simulated years do not create evidence about control effectiveness. Use **120–160 words**.

### 1a. Evidence audit
| Item | Classification | Evidence to seek / explanation |
|---|---|---|
| 18% annual incident probability | … | … |
| Independence of outage and secondary response | … | … |
| 1% risk limit | … | … |

In [ ]:
def calibration_metrics(observed, lower, upper):
    """Return {'coverage': proportion, 'misses': integer, 'mean_width_hours': value}."""
    observed, lower, upper = map(lambda x: np.asarray(x, dtype=float),
                                (observed, lower, upper))
    if not (observed.shape == lower.shape == upper.shape) or len(observed) == 0:
        raise ValueError('Use equally sized, nonempty arrays.')
    if np.any(lower > upper):
        raise ValueError('Lower bounds cannot exceed upper bounds.')
    # YOUR ANALYSIS CODE: identify interval hits, then calculate the three metrics.
    raise NotImplementedError('Complete calibration_metrics.')

# PROVIDED: table construction and display.
def run_calibration():
    rows = []
    for name in ('E1', 'E2'):
        result = calibration_metrics(calibration['observed_hours'],
                                     calibration[name + '_lower'], calibration[name + '_upper'])
        rows.append({'estimator': name, **result})
    table = pd.DataFrame(rows).set_index('estimator')
    display(table.style.format({'coverage': '{:.0%}', 'misses': '{:.0f}',
                                'mean_width_hours': '{:.2f}'}))
    return table
calibration_summary = run_task('Part 1b', run_calibration)

### 1c. Your interpretation
*Write about 120–160 words. Refer to your calculated results.*

…

### Short definitions for implementation and validation
- **Analytical benchmark:** the expected value calculated directly from the model and distribution formulas, rather than estimated from random draws.
- **Deterministic test:** a reproducible check with a predictable answer. Check a mathematical relationship that should hold, rather than requiring one random frequency to equal its population probability exactly.
- **Random seed:** the starting state of the random-number generator. Reusing it reproduces the same draws; it does not provide new evidence or improve assumptions.

## 2. Implement and validate the loss model [25 marks]
One trial represents **one possible year**, with at most one material incident. For each strategy, $I$ is its incident indicator; $R$ is incident-response cost; $D$ is outage duration in days; and $J$ indicates additional data-exposure response. Costs are non-overlapping. Let $C$ be the annual incremental control cost.

$$
S=R+c_dD+c_sJ,\qquad L=IS,\qquad T=L+C,
$$

where $c_d=30{,}000$ CAD/day, $c_s=300{,}000$ CAD, $S$ is potential **conditional incident loss**, $L$ is **annual incident loss**, and $T$ is annual total cost in this model.

Sample $I$ with annual probability $p$. Sample $R$ from the supplied triangular distribution and $D$ from the supplied lognormal quantiles. In the baseline, $I,R,D,J$ are independent and $P(J=1)=q=0.20$. Because severity is independent of $I$, this is also $P(J=1\mid I=1)$. Trials are independent. Part 5 changes only the dependence between $D$ and $J$.

**2a [14].** Complete the analytical body of `simulate_years`: generate the incident indicator, response cost, outage duration, and secondary indicator, then compute conditional and annual loss. Validation, generator setup, output-column construction, and the call pattern are provided. Use the supplied `secondary_probabilities` helper; complete its stress branch in Part 5. Return **all six columns**. Retain potential severity even in no-incident rows, but set annual incident loss to zero for those rows. Annual control cost is not part of this function's `annual_loss` output.

**2b [5].** Run the provided checks, then add **one non-duplicative deterministic risk-model test** with a comment stating the error it catches. For example, test a structural relationship among inputs and losses, not merely a printed table. Do not remove the supplied tests.

**2c [6].** Complete `analytic_eal` using the distribution means below. The provided runner simulates 100,000 years per strategy and constructs the validation table. Use the indicated seeds, not repeated seed searches for a favourable answer.

$$
\mathbb{E}[L]=p\left(\mathbb{E}[R]+c_d\mathbb{E}[D]+q c_s\right),
$$
$$
\mathbb{E}[R]=\frac{a+m+b}{3},\qquad
\mathbb{E}[D]=\exp\left(\mu+\frac{\sigma^2}{2}\right).
$$

The diagnostic is the simulated-minus-analytic difference divided by the Monte Carlo standard error. Agreement within about four standard errors is useful, not a proof. Explain briefly **one numerical agreement and one limitation** in the response cell.

### Provided helper: lognormal parameters from quantiles
The median $d_{50}$ and 90th percentile $d_{90}$ are **not** the log-scale mean and standard deviation used by NumPy. With $z_{0.90}=\Phi^{-1}(0.90)$:

$$
\mu=\ln(d_{50}),\qquad
\sigma=\frac{\ln(d_{90})-\ln(d_{50})}{z_{0.90}}.
$$

Use `rng.lognormal(mu, sigma, size=n)`. Do not truncate at $d_{90}$. Triangular endpoints are hard limits within this model; a lognormal percentile is not.

In [ ]:
def lognormal_parameters(p50, p90):
    """Return log-scale (mu, sigma) for positive p50 < p90."""
    p50, p90 = float(p50), float(p90)
    if not (np.isfinite(p50) and np.isfinite(p90) and 0 < p50 < p90):
        raise ValueError('Require finite quantiles with 0 < p50 < p90.')
    mu = np.log(p50)
    sigma = (np.log(p90) - mu) / NormalDist().inv_cdf(0.90)
    return mu, sigma

### Provided distribution overview - read it as in Tutorial 2
The three panels show the distributions specified by the supplied inputs, before any loss simulation. Bernoulli bar heights are probabilities; triangular and lognormal curve heights are **densities**, so probability is area over a range.

The P90 duration markers are percentiles, not maxima: the lognormal distribution continues beyond them. Current and prevention share the same severity distributions; prevention changes incident probability. This figure is supplied to help you read the model and carries no separate marks.

In [ ]:
# PROVIDED: displays input distributions; does not complete the assessed sampler.
PLOT_COLORS = {'current': '#b5122b', 'prevention': '#2662a3', 'recovery': '#177245'}

def plot_input_distributions():
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    x = np.arange(2)
    width = 0.24
    for i, name in enumerate(STRATEGIES):
        p = float(parameters.loc[name, 'p_incident'])
        axes[0].bar(x + (i - 1) * width, [1 - p, p], width=width,
                    color=PLOT_COLORS[name], label=name.title())
    axes[0].set(xticks=x, xticklabels=['No incident', 'Incident'],
                ylabel='Probability', title='Bernoulli: annual occurrence', ylim=(0, 1))
    axes[0].yaxis.set_major_formatter(PercentFormatter(1))
    axes[0].legend(fontsize=8)

    # Current and prevention have identical cost and duration distributions.
    for name, label in [('current', 'Current / prevention'), ('recovery', 'Recovery')]:
        row = parameters.loc[name]
        a, m, b = [float(row[c]) for c in ['response_min', 'response_mode', 'response_max']]
        cost = np.linspace(0, 90_000, 1000)
        density = np.zeros_like(cost)
        left, right = (cost >= a) & (cost <= m), (cost > m) & (cost <= b)
        density[left] = 2 * (cost[left] - a) / ((b - a) * (m - a))
        density[right] = 2 * (b - cost[right]) / ((b - a) * (b - m))
        axes[1].plot(cost / 1000, density * 1000, color=PLOT_COLORS[name], label=label)

        duration = np.linspace(0.025, 25, 1500)
        mu, sigma = lognormal_parameters(row['outage_p50'], row['outage_p90'])
        duration_density = np.exp(-0.5 * ((np.log(duration) - mu) / sigma)**2)
        duration_density /= duration * sigma * np.sqrt(2 * np.pi)
        axes[2].plot(duration, duration_density, color=PLOT_COLORS[name], label=label)
        axes[2].axvline(row['outage_p90'], color=PLOT_COLORS[name], linestyle=':', alpha=0.7)

    axes[1].set(xlabel='Response cost (CAD thousands)', ylabel='Density per CAD 1,000',
                title='Triangular: response cost', xlim=(0, 90), ylim=(0, None))
    axes[2].set(xlabel='Outage duration (days)', ylabel='Density per day',
                title='Lognormal: outage duration', xlim=(0, 25), ylim=(0, None))
    axes[1].legend(fontsize=8)
    axes[2].legend(fontsize=8)
    axes[2].text(0.97, 0.93, 'Dotted lines: P90', transform=axes[2].transAxes,
                 ha='right', va='top', fontsize=8)
    for ax in axes:
        ax.grid(axis='y', alpha=0.2)
    fig.tight_layout()
    plt.show()

plot_input_distributions()

In [ ]:
def secondary_probabilities(days, d90, dependence):
    """Probability of additional response for each potential incident severity."""
    days = np.asarray(days, dtype=float)
    if dependence == 'independent':
        return np.full(days.shape, SECONDARY_PROB)
    if dependence != 'dependent':
        raise ValueError('Use independent or dependent.')
    # PART 5 ONLY: derive q_low and return probabilities using the population d90.
    # Preserve the overall secondary probability. Do not estimate d90 from this sample.
    raise NotImplementedError('Complete the dependence test in Part 5.')

In [ ]:
def simulate_years(strategy, n, seed, dependence='independent', p_override=None):
    """Return potential severity and actual annual incident loss, one row per year."""
    # PROVIDED: argument checks and a local random-number generator.
    if strategy not in parameters.index:
        raise ValueError('Unknown strategy.')
    if not isinstance(n, (int, np.integer)) or n < 1:
        raise ValueError('n must be a positive integer.')
    if dependence not in ('independent', 'dependent'):
        raise ValueError('Unknown dependence setting.')
    row = parameters.loc[strategy]
    p = float(row['p_incident'] if p_override is None else p_override)
    if not 0 <= p <= 1:
        raise ValueError('Incident probability must lie in [0, 1].')
    rng = np.random.default_rng(seed)
    mu, sigma = lognormal_parameters(row['outage_p50'], row['outage_p90'])

    # YOUR CORE MODEL CODE:
    # 1. Set incident using p and a Bernoulli trial for each year.
    # 2. Set response using response_min, response_mode, and response_max.
    # 3. Set days using the lognormal helper's mu and sigma.
    # 4. Set q using secondary_probabilities(days, row['outage_p90'], dependence).
    # 5. Set secondary using q; compute conditional and annual losses.
    # Names to define: incident, response, days, secondary, conditional, annual.
    # For reproducible comparison, make random draws in the listed order.
    raise NotImplementedError('Complete the central loss-model calculations.')

    # PROVIDED: return structure (keep these column names).
    return pd.DataFrame({
        'incident': incident, 'response_cost': response, 'outage_days': days,
        'secondary': secondary, 'conditional_loss': conditional, 'annual_loss': annual
    })

In [ ]:
# Provided basic tests. Keep these in your submission.
def run_basic_tests():
    required = {'incident', 'response_cost', 'outage_days', 'secondary',
                'conditional_loss', 'annual_loss'}
    sample = simulate_years('current', 2000, SEED)
    assert len(sample) == 2000 and required.issubset(sample.columns)
    assert np.isfinite(sample[list(required)].to_numpy()).all()
    assert (sample[['response_cost', 'outage_days', 'conditional_loss', 'annual_loss']] >= 0).all().all()
    assert set(sample['incident'].unique()) <= {0, 1}
    assert set(sample['secondary'].unique()) <= {0, 1}
    assert (sample.loc[sample['incident'] == 0, 'annual_loss'] == 0).all()
    assert np.allclose(sample['annual_loss'], sample['incident'] * sample['conditional_loss'])
    repeat = simulate_years('current', 2000, SEED)
    pd.testing.assert_frame_equal(sample, repeat)
    zero = simulate_years('current', 2000, SEED, p_override=0)
    one = simulate_years('current', 2000, SEED, p_override=1)
    assert (zero['annual_loss'] == 0).all()
    assert (one['incident'] == 1).all()
    assert np.allclose(one['annual_loss'], one['conditional_loss'])
    print('Provided baseline tests passed. This checks code, not model validity.')

try:
    run_basic_tests()
except NotImplementedError as exc:
    print(f'NOT COMPLETED: {exc}')

In [ ]:
# 2b. YOUR CODE HERE: one additional deterministic test.
# Add a short comment explaining the error it would catch.

In [ ]:
def analytic_eal(strategy):
    """Expected annual incident loss in CAD; exclude the annual control cost."""
    row = parameters.loc[strategy]
    mu, sigma = lognormal_parameters(row['outage_p50'], row['outage_p90'])
    # YOUR CODE: calculate the response mean, duration mean, and expected annual loss.
    # Use row['p_incident'], DAY_COST, SECONDARY_PROB, and SECONDARY_COST.
    raise NotImplementedError('Complete analytic_eal.')

In [ ]:
# PROVIDED: run the three baseline experiments and construct the diagnostic table.
baseline, validation = None, None

def run_baseline():
    samples, rows = {}, []
    for i, strategy in enumerate(STRATEGIES):
        sample = simulate_years(strategy, N_SIM, SEED + i)
        samples[strategy] = sample
        loss = sample['annual_loss'].to_numpy()
        analytic = analytic_eal(strategy)
        mc_mean = loss.mean()
        mc_se = loss.std(ddof=1) / np.sqrt(len(loss))
        rows.append({'strategy': strategy, 'analytic_eal': analytic,
                     'simulated_eal': mc_mean, 'mc_se': mc_se,
                     'z_mc': (mc_mean - analytic) / mc_se})
    table = pd.DataFrame(rows).set_index('strategy')
    display(table.style.format('{:,.2f}'))
    return samples, table

result = run_task('Part 2c', run_baseline)
if result is not None:
    baseline, validation = result

### 2c. Validation interpretation
*In 40–60 words, use the validation table to describe one agreement and explain why it does not establish the realism of the assumptions.*

…

### Short definitions for Part 3
- **Net expected benefit (NEB):** current expected total cost minus the option's expected total cost. A positive NEB is a saving under this model.
- **Affordable:** annual control cost is at most the budget. **Eligible / feasible:** the option satisfies both the budget and the stated incident-loss risk limit.
- **Exceedance probability:** the fraction of all modelled years with incident loss strictly above a threshold. It is a probability, not a density.
- **Point mass at zero / zero inflation:** many annual outcomes are exactly zero because no incident occurs. A zero median does not rule out costly incident years.

## 3. Compare controls, not just averages [20 marks]
**3a [8].** Complete `loss_summary` and `assess_controls`. Summarize **all simulated years**, including zero-loss years. Report the mean, median, 95th and 99th percentiles, and $P(L>400{,}000)$. Complete expected total cost, net expected benefit, affordability, and the risk-limit flag; return the preferred affordable strategy by expected cost alone and the strategy permitted by the full policy. Return `None` when no option meets the policy. Do not select an unaffordable option.

For strategy $k$ and current controls $0$:

$$
\mathbb{E}[T_k]=\mathbb{E}[L_k]+C_k,
$$
$$
\operatorname{NEB}_k=\left(\mathbb{E}[L_0]+C_0\right)-\left(\mathbb{E}[L_k]+C_k\right).
$$

Here $C_0=0$, so the current strategy's net expected benefit is zero. Apply the CAD 20,000 budget and **incident-loss**, not total-cost, exceedance limit. Use full precision for decisions; round only for display.

**3b [6].** Complete `exceedance_probability`. The plotting code is supplied; run it to display the annual-loss exceedance curves, expected annual total cost by strategy, and the supplied conditional-severity histogram. Use strict `>` for every exceedance calculation, a common threshold grid, and **all years in the denominator**. Read from the curves how the choices differ near the policy threshold and explain that difference in 3c.

$$
\widehat{P}(L>x)=\frac{1}{N}\sum_{i=1}^{N}\mathbf{1}\{L_i>x\}.
$$

**3c [6].** In **120–160 words**, compare the cost-only choice with the full-policy choice using numerical results and the curves. Explain why a zero median does not mean negligible risk. An exceedance curve is not a probability density.

In [ ]:
def loss_summary(losses, threshold=LOSS_THRESHOLD):
    """Return mean, median, p95, p99, and p_exceed (a proportion, not a percentage)."""
    losses = np.asarray(losses, dtype=float)
    if losses.ndim != 1 or len(losses) == 0 or not np.isfinite(losses).all():
        raise ValueError('Use a nonempty finite one-dimensional annual-loss array.')
    # YOUR CORE RISK CODE: compute all five metrics, including the zero-loss years.
    raise NotImplementedError('Complete loss_summary.')

In [ ]:
def assess_controls(summary, control_costs):
    """Return (decision_table, cost_only_choice, policy_choice).

    summary: DataFrame indexed by strategy, with the five loss_summary columns.
    control_costs: Series of annual costs, indexed by the same strategy names.
    A choice is a strategy name, or None when the relevant feasible set is empty.
    """
    table = summary.copy()
    table['control_cost'] = control_costs.reindex(table.index)
    # YOUR DECISION CODE: add these columns:
    # expected_total_cost, net_expected_benefit, affordable, meets_risk_limit, eligible.
    # Select cost_only_choice among affordable strategies.
    # Select policy_choice among strategies meeting BOTH budget and risk limit.
    # Do not select using rounded values; handle an empty eligible set explicitly.
    raise NotImplementedError('Complete the control-comparison and selection logic.')
    return table, cost_only_choice, policy_choice

# PROVIDED: result collection and readable formatting.
def display_decision(table):
    formats = {c: '{:,.0f}' for c in ['mean','median','p95','p99','control_cost',
                                     'expected_total_cost','net_expected_benefit']}
    formats['p_exceed'] = '{:.3%}'
    display(table.style.format(formats))

def decision_from_samples(samples):
    summary = pd.DataFrame.from_dict(
        {name: loss_summary(samples[name]['annual_loss']) for name in STRATEGIES},
        orient='index')
    summary.index.name = 'strategy'
    table, cost_choice, policy_choice = assess_controls(summary, parameters['annual_control_cost'])
    display_decision(table)
    print('Cost-only choice:', cost_choice)
    print('Full-policy choice:', policy_choice if policy_choice is not None else 'NO FEASIBLE OPTION')
    return table

decision_table = None
if baseline is None:
    print('NOT COMPLETED — finish Part 2 before running Part 3.')
else:
    decision_table = run_task('Part 3a', lambda: decision_from_samples(baseline))

In [ ]:
def exceedance_probability(losses, threshold):
    """Return the fraction of ALL annual losses strictly above a scalar threshold."""
    losses = np.asarray(losses, dtype=float)
    # YOUR CODE: return the empirical exceedance probability.
    raise NotImplementedError('Complete exceedance_probability.')

# PROVIDED: plotting scaffold. The estimate passed to it comes from your function.
def plot_exceedance(samples):
    thresholds = np.linspace(0, 1_200_000, 241)
    curves = {name: [exceedance_probability(samples[name]['annual_loss'], x)
                     for x in thresholds] for name in STRATEGIES}
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for name in STRATEGIES:
        ax.plot(thresholds, curves[name], label=name.title())
    ax.axvline(LOSS_THRESHOLD, linestyle='--', linewidth=1, label='CAD 400,000 threshold')
    ax.axhline(RISK_LIMIT, linestyle=':', linewidth=1, label='1% annual limit')
    ax.set(xlabel='Annual incident-loss threshold (CAD)',
           ylabel='Probability annual incident loss exceeds threshold',
           title='Baseline annual-loss exceedance curves', xlim=(0, 1_200_000))
    ax.xaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.legend()
    fig.tight_layout()
    plt.show()

if baseline is not None:
    run_task('Part 3b', lambda: plot_exceedance(baseline))
else:
    print('NOT COMPLETED — baseline samples are needed for the curves.')

In [ ]:
# PROVIDED: second figure, using the decision quantities computed in Part 3a.
if decision_table is not None:
    fig, ax = plt.subplots(figsize=(6.8, 4))
    ax.bar(decision_table.index.str.title(), decision_table['expected_total_cost'])
    ax.set(ylabel='Expected annual total cost (CAD)',
           title='Incident loss plus annual control cost')
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    fig.tight_layout()
    plt.show()
else:
    print('NOT COMPLETED — finish the control comparison first.')

### Provided conditional-severity histogram - compare with Tutorial 2
This third Part 3 figure shows **loss given that an incident occurs**. It therefore excludes no-incident years, while the annual summaries and exceedance curves include them.

Use the histogram to inspect severity and the right tail. Read probability from area across a range, not the height at one exact loss. The visible range ends at the pooled 99.5th percentile of incident losses; omitted tail values remain in every calculation. No extra histogram code or written answer is required.

In [ ]:
# PROVIDED: histogram after you complete the annual-loss model.
def plot_incident_severity(samples):
    incident_losses = {
        name: samples[name].loc[samples[name]['incident'] == 1, 'annual_loss'].to_numpy()
        for name in STRATEGIES
    }
    nonempty = [values for values in incident_losses.values() if len(values)]
    if not nonempty:
        print('No incident years were sampled; conditional severity is unavailable.')
        return
    visible_upper = float(np.quantile(np.concatenate(nonempty), 0.995))
    bins = np.linspace(0, visible_upper / 1000, 61)
    bin_width = bins[1] - bins[0]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for name in STRATEGIES:
        values = incident_losses[name] / 1000
        if len(values):
            # Normalize by ALL incident draws, including values beyond the visible range.
            weights = np.full(len(values), 1 / (len(values) * bin_width))
            ax.hist(values, bins=bins, weights=weights, histtype='step', linewidth=1.8,
                    color=PLOT_COLORS[name], label=name.title())
    ax.set(xlabel='Incident loss given an incident (CAD thousands)',
           ylabel='Density per CAD 1,000', title='Severity in incident years',
           xlim=(0, visible_upper / 1000), ylim=(0, None))
    ax.text(0.98, 0.94, 'No-incident years excluded here', transform=ax.transAxes,
            ha='right', va='top', fontsize=9)
    ax.legend()
    ax.grid(axis='y', alpha=0.2)
    fig.tight_layout()
    plt.show()

if baseline is not None:
    run_task('Part 3 - provided histogram', lambda: plot_incident_severity(baseline))
else:
    print('NOT COMPLETED - baseline samples are needed for the severity histogram.')

### 3c. Your baseline recommendation
*Write about 120–160 words. Cite your table/figures and distinguish incident loss from total cost.*

…

## 4. Numerical precision is not evidence [15 marks]
**4a [8].** Complete `precision_summary` to compute the mean, sample standard deviation, Monte Carlo standard error, approximate 95% Monte Carlo interval for the mean, and the 5th–95th percentile **annual-loss** interval. Include both widths. The supplied loop runs three seeds at each of $N=1{,}000$, $10{,}000$, and $100{,}000$ and displays a nine-row table. Use `ddof=1` for the sample standard deviation.

$$
\bar L=\frac{1}{N}\sum_{i=1}^{N}L_i,\qquad
\widehat{\operatorname{SE}}_{\mathrm{MC}}=\frac{s_L}{\sqrt{N}},
$$
$$
\operatorname{CI}_{\mathrm{MC},95\%}\approx
\left[\bar L-1.96\frac{s_L}{\sqrt{N}},\;\bar L+1.96\frac{s_L}{\sqrt{N}}\right].
$$

The annual-outcome interval is $[\widehat Q_{0.05}(L),\widehat Q_{0.95}(L)]$. It describes the 5th–95th percentile range of simulated annual outcomes, **not** a confidence interval for the mean. Because annual losses can have a point mass at zero, this interval need not contain exactly 90% of outcomes. Run the supplied Monte Carlo stabilization plot after completing `precision_summary`; no additional plotting code is required.

**4b [7].** In **100–140 words**, use the table and stabilization plot to explain what becomes more stable as $N$ grows, whether either width must decrease monotonically in these runs, and the expected standard-error change when $N$ increases by a factor of 100. Correct the claim: “We simulated 100,000 years, so our input assumptions are now well measured.” Both intervals are conditional on the selected model.

### Short definitions and reading guide for Part 4
- **Sample standard deviation (`sample_sd`):** the spread of sampled annual losses. `ddof=1` uses $N-1$ in the variance denominator.
- **Monte Carlo standard error:** estimated numerical sampling error of the mean, $s_L/\sqrt{N}$.
- **Approximate 95% Monte Carlo interval:** the mean plus/minus 1.96 standard errors under the fixed model; its width is upper minus lower endpoint.
- **Annual-outcome interval:** the range from the sampled P05 to P95 of annual loss. It describes variation in outcomes, rather than precision of the mean. A point mass at zero means it need not contain exactly 90% of sampled years.
- **Monotonically decreasing:** smaller at every successive step, with no temporary increase. Random fluctuations can prevent an observed width from decreasing at every step.

The supplied stabilization plot uses prefixes of the same current-controls simulation. Its line is the running mean, its shaded band is the approximate 95% Monte Carlo interval, and its dashed line is your analytical benchmark. Each checkpoint reuses the same model, rather than adding real incident observations. The horizontal axis is logarithmic: equal spacing represents multiplicative changes in $N$.

In [ ]:
def precision_summary(losses):
    """Return mean, sample_sd, mc_se, mc_low, mc_high, mc_width, outcome_p05, outcome_p95, outcome_width."""
    losses = np.asarray(losses, dtype=float)
    n = len(losses)
    if n < 2 or losses.ndim != 1 or not np.isfinite(losses).all():
        raise ValueError('Use at least two finite annual-loss values.')
    # YOUR ANALYSIS CODE: calculate numerical uncertainty AND annual-outcome variability.
    # Return a dictionary with exactly the keys described above.
    raise NotImplementedError('Complete precision_summary.')

# PROVIDED: repeated experiments, seed management, table construction, and display.
def run_precision_experiment():
    rows = []
    for i, n in enumerate([1_000, 10_000, 100_000]):
        for r in range(3):
            seed = SEED + 100 + 10*i + r
            sample = simulate_years('current', n, seed)
            rows.append({'N': n, 'seed': seed, **precision_summary(sample['annual_loss'])})
    table = pd.DataFrame(rows)
    display(table.style.format({c: '{:,.1f}' for c in table.columns if c not in ['N','seed']}))
    return table
precision_table = run_task('Part 4a', run_precision_experiment)

In [ ]:
# PROVIDED: the visual calculation uses YOUR completed precision_summary function.
def plot_mc_stabilization(samples):
    losses = samples['current']['annual_loss'].to_numpy()
    checkpoints = np.unique(np.logspace(2, np.log10(len(losses)), 70).astype(int))
    rows = [{'N': int(n), **precision_summary(losses[:n])} for n in checkpoints]
    convergence = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(9, 4.6))
    ax.plot(convergence['N'], convergence['mean'], color=PLOT_COLORS['current'],
            linewidth=1.8, label='Running mean annual loss')
    ax.fill_between(convergence['N'], convergence['mc_low'], convergence['mc_high'],
                    color=PLOT_COLORS['current'], alpha=0.18,
                    label='Approx. 95% Monte Carlo interval')
    ax.axhline(analytic_eal('current'), color='#333333', linestyle='--',
               linewidth=1.3, label='Analytical expected annual loss')
    ax.set_xscale('log')
    ax.set(xlabel='Number of simulated years (log scale)',
           ylabel='Estimated mean annual incident loss (CAD)',
           title='Monte Carlo stabilization under a fixed model')
    ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))
    ax.grid(alpha=0.2)
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
    return convergence

convergence_table = None
if baseline is not None and precision_table is not None:
    convergence_table = run_task('Part 4 - stabilization plot',
                                 lambda: plot_mc_stabilization(baseline))
else:
    print('NOT COMPLETED - finish the loss model and precision_summary for stabilization.')

### 4b. Your explanation
*Write about 100–140 words, referring to the observed table and stabilization plot.*

…

### Short definitions for the new probability terms in Part 5
- **Conditional probability**, $P(J=1\mid D>d_{90})$: the probability of additional response within the group of outages longer than $d_{90}$. The vertical bar means "given that".
- **Marginal probability / distribution:** the overall probability or distribution of one variable considered on its own, without restricting to a subgroup. Preserving marginals does not preserve how two variables occur together.
- **Dependence:** knowing one variable changes the probabilities for another. Dependence describes a relationship; it does not establish that one variable causes the other.
- **Law of total probability:** overall probability is the weighted average of subgroup probabilities. If a subgroup has proportion $w$, then $P(J=1)=wq_{\mathrm{high}}+(1-w)q_{\mathrm{low}}$.
- **Population percentile:** a percentile of the specified model distribution, supplied here as $d_{90}$. A **sample percentile** is estimated from one finite set of draws and fluctuates. Use the supplied population value in the stress rule.
- **Linearity of expectation:** $E[A+B]=E[A]+E[B]$, including when $A$ and $B$ are dependent. The distribution of a sum still depends on how its components occur together.

**Small probability example, separate from this case:** Suppose 25% of devices are in group A, where the fault probability is 10%, and 75% are in group B, where it is 2%. The overall fault probability is $0.25(0.10)+0.75(0.02)=0.04$, or 4%. Use the same weighting idea to derive the unknown probability in Part 5; the assignment's numerical derivation remains your task.

## 5. Challenge one assumption, then advise the client [25 marks]
A responder suggests that long outages and additional data-exposure response may coincide in more severe incidents. This is a **hypothetical stress model**, not a newly observed fact and not evidence that downtime causes data exposure.

**5a [10].** Complete only the stress branch of `secondary_probabilities` above. Use each strategy's **supplied population** 90th-percentile duration, not a sample percentile. The changed relationship is

$$
P(J=1\mid D>d_{90})=0.80,\qquad
P(J=1\mid D\leq d_{90})=q_{\mathrm{low}}.
$$

Since $P(D>d_{90})=0.10$, derive $q_{\mathrm{low}}$ from

$$
0.10(0.80)+0.90q_{\mathrm{low}}=0.20.
$$

Show the algebra in the prediction Markdown cell, then implement the conditional probabilities. Keep the individual distributions of $R$ and $D$, all costs, the incident probability, and the marginal probability of $J$ unchanged. $I$ remains independent of severity; $R$ remains independent of $D$ and $J$. The supplied experiment loop runs 100,000 years per strategy and constructs the sampler-check and six-row comparison tables. Inspect sample proportions of $D>d_{90}$ and $J=1$ using all potential-severity rows; they should be close to 0.10 and 0.20, not exactly equal.

**5b [7].** **Before running** the stress model, predict in **40–60 words** whether analytic EAL changes, and justify your prediction. Afterwards, use the tables and supplied comparison curves to interpret the observed changes in **100–140 words**. Does the same mean imply the same risk profile? Must every percentile or exceedance probability increase? Does any option still satisfy the policy? Do not mistake Monte Carlo fluctuations for a change in analytic expectation, or row-wise simulation differences for measured causal effects.

**5c [8].** Write a **200–250-word management memo in the Markdown cell below**. State the baseline recommendation, what the stress model changes, and any unresolved policy conflict. Refer to at least three numerical results and one relevant textbook idea. Include one limitation or nonfinancial concern. Request one specific measurement: state what would be observed, which assumption it informs, and how the result could affect the decision. Do not describe the stress model as proof that a real control fails.

**The memo stays inside this notebook. There is no separate report or PDF submission.**

### 5b. Prediction — write before running Part 5
*About 40–60 words. Show your calculation of $q_{\mathrm{low}}$ and predict the mean.*

…

In [ ]:
# PROVIDED: complete the stress branch above, rerun its definition, then run this cell.
stressed, sampler_checks, comparison_table = None, None, None

def run_stress_experiment():
    if decision_table is None:
        print('NOT COMPLETED — finish the baseline decision table first.')
        return None
    samples, checks = {}, []
    for i, strategy in enumerate(STRATEGIES):
        sample = simulate_years(strategy, N_SIM, SEED + i, dependence='dependent')
        samples[strategy] = sample
        checks.append({'strategy': strategy,
                       'fraction_above_population_d90':
                           (sample['outage_days'] > parameters.loc[strategy, 'outage_p90']).mean(),
                       'fraction_secondary': sample['secondary'].mean()})
    checks = pd.DataFrame(checks).set_index('strategy')
    display(checks.style.format('{:.3%}'))
    stress_table = decision_from_samples(samples)
    cols = ['mean','expected_total_cost','p99','p_exceed','eligible']
    comparison = pd.concat({'independent': decision_table[cols],
                            'dependent': stress_table[cols]}, names=['model','strategy'])
    display(comparison.style.format({'mean':'{:,.0f}','expected_total_cost':'{:,.0f}',
                                     'p99':'{:,.0f}','p_exceed':'{:.3%}'}))
    return samples, checks, comparison

result = run_task('Part 5a', run_stress_experiment)
if result is not None:
    stressed, sampler_checks, comparison_table = result

### Provided baseline-versus-stress exceedance curves
The solid curve in each panel is the independent baseline; the dashed curve is the hypothetical dependent model. Use the common axes to compare each strategy near the CAD 400,000 threshold and 1% annual limit. These are **annual** probabilities, so all simulated years remain in the denominator. The plot supports your existing Part 5 interpretation; it introduces no separate written task.

In [ ]:
# PROVIDED: uses your exceedance_probability calculation for both models.
def plot_stress_comparison(base_samples, stress_samples):
    thresholds = np.linspace(0, 1_200_000, 241)
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharex=True, sharey=True)
    for ax, name in zip(axes, STRATEGIES):
        for label, samples, style in [('Independent baseline', base_samples, '-'),
                                      ('Dependent stress model', stress_samples, '--')]:
            probabilities = [exceedance_probability(samples[name]['annual_loss'], x)
                             for x in thresholds]
            ax.plot(thresholds / 1000, probabilities, linestyle=style,
                    color=PLOT_COLORS[name], linewidth=1.8, label=label)
        ax.axvline(LOSS_THRESHOLD / 1000, color='#666666', linestyle=':', linewidth=1)
        ax.axhline(RISK_LIMIT, color='#666666', linestyle=':', linewidth=1)
        ax.set(title=name.title(), xlabel='Loss threshold (CAD thousands)',
               xlim=(0, 1200), ylim=(0, None), xticks=[0, 400, 800, 1200])
        ax.yaxis.set_major_formatter(PercentFormatter(1))
        ax.grid(alpha=0.15)
    axes[0].set_ylabel('P(annual incident loss > threshold)')
    axes[1].legend(loc='upper right', fontsize=8)
    fig.suptitle('Changing dependence: compare the annual-loss tails', fontsize=12)
    fig.tight_layout()
    plt.show()

if baseline is not None and stressed is not None:
    run_task('Part 5 - provided comparison plot',
             lambda: plot_stress_comparison(baseline, stressed))
else:
    print('NOT COMPLETED - baseline and stress samples are needed for this plot.')

### 5b. Your interpretation after the experiment
*About 100–140 words. Use your results; do not just restate the prediction.*

…

### 5c. Final memo to the Executive Director
*Write 200–250 words here, in Markdown. Include a short textbook citation, such as “Hubbard and Seiersen, 2023, Chapter 2.” This is part of the notebook, not an additional file.*

…

## Final submission check
Restart the kernel, run all cells in order, resolve errors and unfinished tasks, and save with outputs visible. Do not submit enormous raw simulation tables. Use readable rounding in displays but full precision in calculations and decisions. Show percentages clearly and label monetary units.

**Submit only `CMPT318_A1_GroupXX.ipynb`.** All written answers, equations, interpretations, the final memo, contributions, and source acknowledgments remain in Markdown cells. **Do not upload an additional PDF, Pages file, report, CSV, or ZIP.** The marker will run your notebook next to the original `data` folder.

The supplied CSV files are inputs and must remain unchanged. No hard-coded absolute paths, hidden manual data edits, or network downloads are needed. Follow the existing course collaboration, AI-use, and late-submission policies. Be prepared to explain your code and conclusions.

In [ ]:
# PROVIDED: presence checks are not grading or a substitute for reading your answers.
required_outputs = ['calibration_summary', 'baseline', 'validation', 'decision_table',
                    'precision_table', 'convergence_table', 'stressed', 'sampler_checks', 'comparison_table']
missing = [name for name in required_outputs if globals().get(name) is None]
if missing:
    print('NOT COMPLETED — missing results:', ', '.join(missing))
else:
    print('Result objects exist. Review calculations, tests, figures, Markdown answers, and memo.')
    print('Submit ONE file: CMPT318_A1_GroupXX.ipynb, saved with outputs.')

### Contributions, sources, and assistance
**Contributions:** …  
**Sources/assistance used:** …

### Course and textbook sources
[1] CMPT 318, *Tutorial 1: Measuring, Calibrating, and Acting on Cyber Risk*, course notebook, Fall 2026. Calibration, explicit assumptions, expected annual loss, and current/residual risk.

[2] CMPT 318, *From uncertainty to a decision: Monte Carlo cyber-risk analysis*, Tutorial Session 2, course notebook, Fall 2026. Sampling, quantiles, exceedance, simulation error, and analytic validation.

[3] Douglas W. Hubbard and Richard Seiersen, *How to Measure Anything in Cybersecurity Risk*, 2nd edition, Wiley, 2023. Review Chapter 2 (measurement and uncertainty), Chapter 3 (Rapid Risk Audit, return on controls, and loss-exceedance curves), and Chapter 7 (calibrated estimates).

The scenario numbers, calibration records, constrained decision rule, and dependence experiment in this assignment are instructor-designed teaching material. They are not reported empirical findings from the book or tutorials. “Expected annual loss” follows the tutorials; the book also uses “annualized expected loss.”